# Configurable n×n Tic-Tac-Toe AI
## Minimax Algorithm with Alpha-Beta Pruning

This notebook demonstrates a configurable Tic-Tac-Toe AI that uses the Minimax algorithm enhanced with Alpha-Beta pruning. It supports n×n board sizes with full-depth search for 3×3 and depth-limited heuristic search for larger boards.

---

## 1. Setup
Clone the repository and import modules.

In [ ]:
# Uncomment the lines below if running on Google Colab
# !git clone -b claude/tic-tac-toe-ai-game-8TPHb https://github.com/amanverma-wsu/440-Project.git
# %cd 440-Project

In [ ]:
from board import Board
from ai import MinimaxAgent, AlphaBetaAgent, RandomAgent
from heuristic import evaluate_board
import random
import time

print("All modules loaded successfully!")

---
## 2. Board Representation
The `Board` class manages the n×n grid, move validation, and win detection.

In [ ]:
# Create a 3x3 board and explore its features
board = Board(3)
print("Empty 3x3 board:")
print(board)
print(f"\nCurrent player: {board.current_player()}")
print(f"Empty cells: {board.get_empty_cells()}")
print(f"Is terminal: {board.is_terminal()}")

In [ ]:
# Demonstrate moves and win detection
demo = Board(3)
demo.make_move(0, 0, Board.X)
demo.make_move(1, 1, Board.O)
demo.make_move(0, 1, Board.X)
demo.make_move(2, 2, Board.O)
demo.make_move(0, 2, Board.X)  # X wins across top row

print(demo)
print(f"\nWinner: {demo.check_winner()}")
print(f"Is terminal: {demo.is_terminal()}")

---
## 3. Play Against the AI
Play a game by calling `make_move()` for your turns. The AI responds automatically.

**How to play:**
- Modify the `human_move(row, col)` calls below with your chosen coordinates
- Row and column indices start at 0
- Run the cell to see the AI's response
- Re-run with new moves to continue the game

In [ ]:
# Initialize a new game
game_board = Board(3)
ai = AlphaBetaAgent(Board.O)  # AI plays as O

def human_move(row, col):
    """Make a human move and let the AI respond."""
    if game_board.is_terminal():
        print("Game is already over!")
        return
    
    if not game_board.make_move(row, col, Board.X):
        print(f"Invalid move ({row}, {col}). Try again.")
        return
    
    print(f"You played: ({row}, {col})")
    print(game_board)
    print()
    
    if game_board.is_terminal():
        winner = game_board.check_winner()
        print("You win!" if winner == Board.X else "It's a draw!")
        return
    
    # AI responds
    move = ai.get_move(game_board)
    game_board.make_move(move[0], move[1], Board.O)
    print(f"AI played: ({move[0]}, {move[1]})")
    print(f"  Nodes explored: {ai.stats.nodes_explored}")
    print(f"  Time: {ai.stats.elapsed_time:.4f}s")
    print()
    print(game_board)
    
    if game_board.is_terminal():
        winner = game_board.check_winner()
        print()
        if winner == Board.O:
            print("AI wins!")
        elif winner == Board.X:
            print("You win!")
        else:
            print("It's a draw!")

print("Game started! You are X, AI is O.")
print(game_board)
print("\nCall human_move(row, col) to play.")

In [ ]:
# Make your moves here one at a time.
# Change the coordinates and re-run this cell for each turn.
human_move(1, 1)  # Example: play center

In [ ]:
human_move(0, 0)  # Your second move

In [ ]:
human_move(2, 2)  # Your third move

In [ ]:
human_move(0, 2)  # Your fourth move

---
## 4. Experiment 1: Correctness Verification (3×3)
Verify that the AI **never loses** when playing optimally on a 3×3 board.

In [ ]:
random.seed(42)
num_games = 200

for ai_sym, opp_sym, label in [(Board.X, Board.O, "AI as X (first)"),
                                (Board.O, Board.X, "AI as O (second)")]:
    wins, draws, losses = 0, 0, 0
    for _ in range(num_games):
        b = Board(3)
        ai_agent = AlphaBetaAgent(ai_sym)
        rand_agent = RandomAgent(opp_sym)

        while not b.is_terminal():
            current = b.current_player()
            if current == ai_sym:
                move = ai_agent.get_move(b)
            else:
                move = rand_agent.get_move(b)
            b.make_move(move[0], move[1], current)

        winner = b.check_winner()
        if winner == ai_sym:
            wins += 1
        elif winner == opp_sym:
            losses += 1
        else:
            draws += 1

    print(f"{label} ({num_games} games vs Random):")
    print(f"  Wins: {wins}, Draws: {draws}, Losses: {losses}")
    print(f"  Result: {'PASS - AI never lost!' if losses == 0 else 'FAIL'}")
    print()

---
## 5. Experiment 2: Minimax vs Alpha-Beta Node Comparison (3×3)
Compare the number of nodes explored by plain Minimax vs Alpha-Beta pruning.

In [ ]:
random.seed(42)
num_games = 50

results = {}
for name, AgentClass in [("Minimax", MinimaxAgent), ("Alpha-Beta", AlphaBetaAgent)]:
    total_nodes = 0
    total_time = 0.0
    total_moves = 0
    wins, draws, losses = 0, 0, 0

    for _ in range(num_games):
        b = Board(3)
        agent = AgentClass(Board.X)
        rand_agent = RandomAgent(Board.O)

        while not b.is_terminal():
            current = b.current_player()
            if current == Board.X:
                move = agent.get_move(b)
                total_nodes += agent.stats.nodes_explored
                total_time += agent.stats.elapsed_time
                total_moves += 1
            else:
                move = rand_agent.get_move(b)
            b.make_move(move[0], move[1], current)

        winner = b.check_winner()
        if winner == Board.X: wins += 1
        elif winner == Board.O: losses += 1
        else: draws += 1

    avg_nodes = total_nodes / max(total_moves, 1)
    avg_time = total_time / max(total_moves, 1)
    results[name] = {"avg_nodes": avg_nodes, "avg_time": avg_time,
                     "wins": wins, "draws": draws, "losses": losses}

    print(f"{name}:")
    print(f"  Wins: {wins}, Draws: {draws}, Losses: {losses}")
    print(f"  Avg nodes/move: {avg_nodes:.1f}")
    print(f"  Avg time/move:  {avg_time:.6f}s")
    print()

mm = results["Minimax"]["avg_nodes"]
ab = results["Alpha-Beta"]["avg_nodes"]
print(f"Node reduction with Alpha-Beta: {(1 - ab / mm) * 100:.1f}%")
print(f"Speedup: {results['Minimax']['avg_time'] / max(results['Alpha-Beta']['avg_time'], 1e-9):.2f}x")

In [ ]:
# Visualize the comparison
try:
    import matplotlib.pyplot as plt

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    algorithms = ["Minimax", "Alpha-Beta"]
    colors = ["#e74c3c", "#2ecc71"]

    # Node comparison
    nodes = [results[a]["avg_nodes"] for a in algorithms]
    bars = ax1.bar(algorithms, nodes, color=colors)
    ax1.set_ylabel("Average Nodes Explored per Move")
    ax1.set_title("Node Expansions (3×3)")
    for bar, val in zip(bars, nodes):
        ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
                f"{val:.0f}", ha="center", fontsize=11)

    # Time comparison
    times = [results[a]["avg_time"] * 1000 for a in algorithms]  # ms
    bars2 = ax2.bar(algorithms, times, color=colors)
    ax2.set_ylabel("Average Time per Move (ms)")
    ax2.set_title("Computation Time (3×3)")
    for bar, val in zip(bars2, times):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
                f"{val:.2f}", ha="center", fontsize=11)

    plt.tight_layout()
    plt.show()
except ImportError:
    print("Install matplotlib for plots: !pip install matplotlib")

---
## 6. Experiment 3: Depth Limit Impact (4×4 Board)
Analyze how different depth limits affect AI performance on a 4×4 board.

In [ ]:
random.seed(42)
depth_limits = [2, 4, 6, 8]
depth_results = []

print(f"{'Depth':<7} {'Wins':<6} {'Draws':<7} {'Losses':<8} {'Avg Nodes':<14} {'Avg Time (s)':<14}")
print("-" * 56)

for depth in depth_limits:
    total_nodes = 0
    total_time = 0.0
    total_moves = 0
    wins, draws, losses = 0, 0, 0
    num_games = 10

    for _ in range(num_games):
        b = Board(4)
        agent = AlphaBetaAgent(Board.X, depth_limit=depth, heuristic=evaluate_board)
        rand_agent = RandomAgent(Board.O)

        while not b.is_terminal():
            current = b.current_player()
            if current == Board.X:
                move = agent.get_move(b)
                total_nodes += agent.stats.nodes_explored
                total_time += agent.stats.elapsed_time
                total_moves += 1
            else:
                move = rand_agent.get_move(b)
            b.make_move(move[0], move[1], current)

        winner = b.check_winner()
        if winner == Board.X: wins += 1
        elif winner == Board.O: losses += 1
        else: draws += 1

    avg_nodes = total_nodes / max(total_moves, 1)
    avg_time = total_time / max(total_moves, 1)
    depth_results.append({"depth": depth, "wins": wins, "draws": draws,
                          "losses": losses, "avg_nodes": avg_nodes, "avg_time": avg_time})

    print(f"{depth:<7} {wins:<6} {draws:<7} {losses:<8} {avg_nodes:<14.1f} {avg_time:<14.6f}")

In [ ]:
# Visualize depth limit impact
try:
    import matplotlib.pyplot as plt

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    depths = [r["depth"] for r in depth_results]
    avg_nodes = [r["avg_nodes"] for r in depth_results]
    win_rates = [r["wins"] / 10 * 100 for r in depth_results]

    ax1.plot(depths, avg_nodes, "o-", color="#3498db", linewidth=2, markersize=8)
    ax1.set_xlabel("Depth Limit")
    ax1.set_ylabel("Avg Nodes per Move")
    ax1.set_title("Depth Limit vs Node Expansions (4×4)")
    ax1.grid(True, alpha=0.3)

    ax2.bar(depths, win_rates, color="#9b59b6", alpha=0.8, width=1.2)
    ax2.set_xlabel("Depth Limit")
    ax2.set_ylabel("Win Rate (%)")
    ax2.set_title("Depth Limit vs Win Rate (4×4)")
    ax2.set_ylim(0, 105)
    ax2.grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    plt.show()
except ImportError:
    print("Install matplotlib for plots: !pip install matplotlib")

---
## 7. Experiment 4: Board Size Scaling Analysis
Measure how AI performance scales as the board size increases.

In [ ]:
random.seed(42)
configs = [
    (3, None, "full"),
    (4, 6, "6"),
    (5, 4, "4"),
]
scaling_results = []

print(f"{'Size':<7} {'Depth':<7} {'Wins':<6} {'Draws':<7} {'Losses':<8} {'Avg Nodes':<14} {'Avg Time (s)':<14}")
print("-" * 63)

for size, depth, depth_str in configs:
    total_nodes = 0
    total_time = 0.0
    total_moves = 0
    wins, draws, losses = 0, 0, 0
    num_games = 10

    heuristic = evaluate_board if depth else None

    for _ in range(num_games):
        b = Board(size)
        if depth:
            agent = AlphaBetaAgent(Board.X, depth_limit=depth, heuristic=heuristic)
        else:
            agent = AlphaBetaAgent(Board.X)
        rand_agent = RandomAgent(Board.O)

        while not b.is_terminal():
            current = b.current_player()
            if current == Board.X:
                move = agent.get_move(b)
                total_nodes += agent.stats.nodes_explored
                total_time += agent.stats.elapsed_time
                total_moves += 1
            else:
                move = rand_agent.get_move(b)
            b.make_move(move[0], move[1], current)

        winner = b.check_winner()
        if winner == Board.X: wins += 1
        elif winner == Board.O: losses += 1
        else: draws += 1

    avg_nodes = total_nodes / max(total_moves, 1)
    avg_time = total_time / max(total_moves, 1)
    scaling_results.append({"size": size, "depth": depth_str, "wins": wins,
                            "draws": draws, "losses": losses,
                            "avg_nodes": avg_nodes, "avg_time": avg_time})

    print(f"{size}×{size:<5} {depth_str:<7} {wins:<6} {draws:<7} {losses:<8} {avg_nodes:<14.1f} {avg_time:<14.6f}")

In [ ]:
# Visualize scaling
try:
    import matplotlib.pyplot as plt

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    sizes = [f"{r['size']}×{r['size']}" for r in scaling_results]
    avg_nodes = [r["avg_nodes"] for r in scaling_results]
    avg_times = [r["avg_time"] for r in scaling_results]

    ax1.bar(sizes, avg_nodes, color="#e67e22", alpha=0.8)
    ax1.set_xlabel("Board Size")
    ax1.set_ylabel("Avg Nodes per Move")
    ax1.set_title("Board Size vs Node Expansions")
    ax1.grid(True, alpha=0.3, axis="y")

    ax2.bar(sizes, avg_times, color="#1abc9c", alpha=0.8)
    ax2.set_xlabel("Board Size")
    ax2.set_ylabel("Avg Time per Move (s)")
    ax2.set_title("Board Size vs Computation Time")
    ax2.grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    plt.show()
except ImportError:
    print("Install matplotlib for plots: !pip install matplotlib")

---
## 8. Summary

### Key Findings

| Metric | Result |
|--------|--------|
| **Correctness (3×3)** | AI never loses against random opponent |
| **Alpha-Beta Pruning** | Significantly reduces nodes explored vs plain Minimax |
| **Depth Limit** | Higher depth = better play but exponentially more computation |
| **Scalability** | Larger boards require depth-limited search with heuristics |

### Algorithm Complexity
- **Minimax**: O(b^d) where b = branching factor, d = depth
- **Alpha-Beta**: O(b^(d/2)) in best case — effectively doubling searchable depth

### Board Size Strategy
- **3×3**: Full-depth search guarantees optimal play
- **4×4**: Depth-limited (depth=6) with heuristic evaluation
- **5×5**: Depth-limited (depth=4) with heuristic evaluation